# Web Scraping AmbitionBox Companies Data

This notebook scrapes top company details from **AmbitionBox** using `requests`, `BeautifulSoup`, and `pandas`.

> **Note on Updated Tags:** AmbitionBox has updated its HTML structure. The old tags (`company-content-wrapper`, `infoEntity`, `p.rating`) have been updated to modern BEM classes (`companyCardWrapper`, `companyCardWrapper__companyName`, `rating_text`, `companyCardWrapper__interLinking`, `companyCardWrapper__ActionWrapper`). This notebook is fully updated and optimized to handle modern tags and export clean CSV datasets.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import time
import re

## 1. Setting Headers (Preventing HTTP 403 Forbidden)
Websites often block automated scrapers if standard browser request headers are missing. We supply a realistic `User-Agent` and `Accept` headers.

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Referer': 'https://www.google.com/'
}

# Test single page request
url = 'https://www.ambitionbox.com/list-of-companies?page=1'
response = requests.get(url, headers=headers)
print(f'Status Code: {response.status_code}')

In [ ]:
soup = BeautifulSoup(response.text, 'lxml')
# Verify page heading
page_title = soup.find('h1').text.strip() if soup.find('h1') else 'AmbitionBox Companies'
print('Page Title:', page_title)

## 2. Inspecting Modern Container and Tags
- **Company Container:** `<div class="companyCardWrapper">`
- **Company Name:** `<h2 class="companyCardWrapper__companyName">`
- **Rating:** `<div class="rating_text">`
- **Industry & Locations:** `<span class="companyCardWrapper__interLinking">`
- **Actions (Reviews, Salaries, Interviews, Jobs, Benefits):** `<a class="companyCardWrapper__ActionWrapper">`
- **Sentiment Highlights (Highly/Critically Rated For):** `<span class="companyCardWrapper__ratingHeader">` & `<span class="companyCardWrapper__ratingValues">`

In [ ]:
# Find all company cards on the page
companies = soup.find_all('div', class_=lambda c: c and 'companyCardWrapper' in c.split())
print(f'Total company cards found on page 1: {len(companies)}')

## 3. Extracting Data for a Single Page

In [ ]:
name = []
full_name = []
rating = []
reviews = []
salaries = []
interviews = []
jobs = []
benefits = []
industry = []
hq_locations = []
rated_highlight = []
profile_url = []

for card in companies:
    # 1. Company Name
    name_el = card.find('h2', class_=re.compile(r'companyCardWrapper__companyName'))
    name.append(name_el.text.strip() if name_el else np.nan)
    
    # Full Name / Alternate Name
    full_name_meta = card.find('meta', itemprop='alternateName')
    full_name.append(full_name_meta['content'].strip() if full_name_meta and full_name_meta.get('content') else (name_el.text.strip() if name_el else np.nan))
    
    # 2. Rating
    rating_el = card.find('div', class_=re.compile(r'rating_text'))
    rating.append(rating_el.text.strip() if rating_el else np.nan)
    
    # 3. Action Counts (Reviews, Salaries, Interviews, Jobs, Benefits)
    actions = {}
    for action in card.find_all('a', class_=re.compile(r'companyCardWrapper__ActionWrapper')):
        count_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionCount'))
        title_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionTitle'))
        if count_el and title_el:
            actions[title_el.text.strip()] = count_el.text.strip()
            
    reviews.append(actions.get('Reviews', np.nan))
    salaries.append(actions.get('Salaries', np.nan))
    interviews.append(actions.get('Interviews', np.nan))
    jobs.append(actions.get('Jobs', np.nan))
    benefits.append(actions.get('Benefits', np.nan))
    
    # 4. Industry & Locations
    interlink_el = card.find('span', class_=re.compile(r'companyCardWrapper__interLinking'))
    parts = [p.strip() for p in interlink_el.text.strip().split('|')] if interlink_el else []
    industry.append(parts[0] if len(parts) > 0 else np.nan)
    hq_locations.append(parts[1] if len(parts) > 1 else np.nan)
    
    # 5. Rated For Highlights
    rated_val_el = card.find('span', class_=re.compile(r'companyCardWrapper__ratingValues'))
    rated_highlight.append(rated_val_el.text.strip() if rated_val_el else np.nan)
    
    # 6. Profile URL
    url_meta = card.find('meta', itemprop='url')
    profile_url.append(url_meta['content'].strip() if url_meta and url_meta.get('content') else np.nan)

# Build DataFrame for Page 1
df = pd.DataFrame({
    'Company_Name': name,
    'Full_Name': full_name,
    'Rating': rating,
    'Reviews': reviews,
    'Salaries': salaries,
    'Interviews': interviews,
    'Jobs': jobs,
    'Benefits': benefits,
    'Industry': industry,
    'Headquarters_Locations': hq_locations,
    'Rated_For': rated_highlight,
    'Profile_URL': profile_url
})

df.head()

In [ ]:
df.shape

## 4. High-Performance Multi-Page Web Scraping

### Performance Best Practices Applied:
1. **`requests.Session()`**: Reuses TCP connections for significantly faster HTTP requests.
2. **List of Dicts Accumulation**: Avoids deprecated `DataFrame.append()` which slows down exponentially.
3. **Polite Delay (`time.sleep`)**: Prevents IP rate-limiting and blocks.
4. **Robust Error Handling (`try-except`)**: Handles missing fields gracefully without breaking the loop.
5. **Configurable Page Range**: Set `TOTAL_PAGES` to scrape as many pages as desired (e.g., 5 to 50+).

In [ ]:
# Specify the number of pages to scrape (e.g. 5 pages = 100 companies)
START_PAGE = 1
TOTAL_PAGES = 5  # Adjust as needed (e.g. 10, 20, 50)

all_companies = []
session = requests.Session()
session.headers.update(headers)

for page in range(START_PAGE, START_PAGE + TOTAL_PAGES):
    url = f'https://www.ambitionbox.com/list-of-companies?page={page}'
    print(f'Scraping Page {page} of {START_PAGE + TOTAL_PAGES - 1}...', end=' ')
    
    try:
        resp = session.get(url, timeout=12)
        if resp.status_code != 200:
            print(f'Failed (Status {resp.status_code})')
            continue
            
        soup = BeautifulSoup(resp.text, 'lxml')
        cards = soup.find_all('div', class_=lambda c: c and 'companyCardWrapper' in c.split())
        
        for card in cards:
            try:
                name_el = card.find('h2', class_=re.compile(r'companyCardWrapper__companyName'))
                c_name = name_el.text.strip() if name_el else np.nan
                
                full_name_meta = card.find('meta', itemprop='alternateName')
                c_full_name = full_name_meta['content'].strip() if full_name_meta and full_name_meta.get('content') else c_name
                
                rating_el = card.find('div', class_=re.compile(r'rating_text'))
                c_rating = rating_el.text.strip() if rating_el else np.nan
                
                actions = {}
                for action in card.find_all('a', class_=re.compile(r'companyCardWrapper__ActionWrapper')):
                    count_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionCount'))
                    title_el = action.find('span', class_=re.compile(r'companyCardWrapper__ActionTitle'))
                    if count_el and title_el:
                        actions[title_el.text.strip()] = count_el.text.strip()
                        
                interlink_el = card.find('span', class_=re.compile(r'companyCardWrapper__interLinking'))
                parts = [p.strip() for p in interlink_el.text.strip().split('|')] if interlink_el else []
                c_industry = parts[0] if len(parts) > 0 else np.nan
                c_location = parts[1] if len(parts) > 1 else np.nan
                
                rated_val_el = card.find('span', class_=re.compile(r'companyCardWrapper__ratingValues'))
                c_rated_for = rated_val_el.text.strip() if rated_val_el else np.nan
                
                url_meta = card.find('meta', itemprop='url')
                c_url = url_meta['content'].strip() if url_meta and url_meta.get('content') else np.nan
                
                all_companies.append({
                    'Company_Name': c_name,
                    'Full_Name': c_full_name,
                    'Rating': c_rating,
                    'Reviews': actions.get('Reviews', np.nan),
                    'Salaries': actions.get('Salaries', np.nan),
                    'Interviews': actions.get('Interviews', np.nan),
                    'Jobs': actions.get('Jobs', np.nan),
                    'Benefits': actions.get('Benefits', np.nan),
                    'Industry': c_industry,
                    'Headquarters_Locations': c_location,
                    'Rated_For': c_rated_for,
                    'Profile_URL': c_url
                })
            except Exception as item_err:
                continue
                
        print(f'Done ({len(cards)} companies extracted)')
        time.sleep(1)  # Polite crawling delay
    except Exception as e:
        print(f'Error: {e}')

# Construct the final DataFrame
final = pd.DataFrame(all_companies)
print(f'\nScraping Complete! Total companies gathered: {len(final)}')

## 5. Exploring the Final Dataset

In [ ]:
final.head(10)

In [ ]:
final.sample(5)

In [ ]:
final.shape

In [ ]:
final.info()

## 6. Exporting the Scraped Data to CSV
Save the final dataset to a `.csv` file format with UTF-8 encoding.

In [ ]:
output_filename = 'World_Companies_Data.csv'
final.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f'Successfully exported dataset to {output_filename}!')

In [ ]:
# Verification by reading the generated CSV back
saved_df = pd.read_csv(output_filename)
print(f'CSV Verification: {saved_df.shape[0]} rows, {saved_df.shape[1]} columns')
saved_df.head(5)